In [0]:
%sql
-- Which council districts have the fastest/slowest average approval times?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_avg_gap_sub_n_issue_cd AS
WITH cd_averages AS (
    SELECT 
    dd.cd,
    COUNT(*) AS total_permits,
    ROUND(AVG(DATEDIFF(id.full_date, sd.full_date)),2) AS avg_day_gap
    FROM la_lakehouse.gold.fact_permits AS fp
    INNER JOIN la_lakehouse.gold.dim_date AS sd
    ON fp.submitted_date_key = sd.date_key
    INNER JOIN la_lakehouse.gold.dim_date AS id
    ON fp.issue_date_key = id.date_key
    LEFT JOIN la_lakehouse.gold.dim_district AS dd
    ON fp.district_key = dd.district_key
    WHERE fp.issue_date_key IS NOT NULL
    AND fp.submitted_date_key IS NOT NULL
    AND dd.cd IS NOT NULL
    GROUP BY dd.cd
    HAVING total_permits >= 100
)
SELECT 
    cd, 
    total_permits,
    avg_day_gap,
    DENSE_RANK() OVER (ORDER BY avg_day_gap ASC) AS fastest_rank, 
    DENSE_RANK() OVER (ORDER BY avg_day_gap DESC) AS slowest_rank
FROM cd_averages
ORDER BY avg_day_gap ASC;


In [0]:
%sql
-- What percentage of permits sit in each status bucket, and how long do "stuck" permits stay unresolved?
CREATE OR REPLACE VIEW la_lakehouse.gold.vw_status_buckets AS
WITH status_buckets AS (
    SELECT 
        fp.permit_nbr,
        fp.status_desc,
        sd.full_date AS submitted_date,
        fp.status_date,
        CASE 
            WHEN fp.status_desc IN (
                'Permit Finshed', 'Permit Finaled', 'Issued', 'CofO Issued', 'Permit Closed', 
                'CofC Issued', 'Permit Expired', 'CofO Corrected', 'OK for CofC', 
                'Re-Activate Permit', 'Intent to Revoke', 'CofC Corrected', 'OK to Issue CofC', 
                'CofO Superseded', 'Order to Comply Issued', 'Not Required', 'PC Approved', 
                'CofO Reactivated', 'TCO Issued', 'Ready to Issue', 'Permit Extended', 
                'PC Info Complete', 'Refund Denied', 'Withdrawn', 'Cancelled', 'Revoked'
            ) THEN 'RESOLVED'
            WHEN fp.status_desc IN (
                'CofO in Progress', 'Refund in Progress', 'No Progress', 'Intent to Correct CofC', 
                'Plan Check', 'In Review', 'Submitted', 'Information Needed', 'Pending', ''
            ) THEN 'STUCK'
            ELSE 'STUCK' 
        END AS status_category
    FROM la_lakehouse.gold.fact_permits AS fp
    LEFT JOIN la_lakehouse.gold.dim_date AS sd
        ON fp.submitted_date_key = sd.date_key
)
SELECT 
    COALESCE(NULLIF(status_desc, ''), 'Unspecified') AS status_desc,
    status_category,
    COUNT(*) AS total_permits,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS pct_of_total,
    ROUND(AVG(
        CASE 
            WHEN status_category = 'STUCK' AND submitted_date IS NOT NULL 
            THEN DATEDIFF(CURRENT_DATE(), submitted_date)
            ELSE NULL 
        END
    ), 2) AS avg_days_stuck_since_submission,
    ROUND(AVG(
        CASE 
            WHEN status_date IS NOT NULL 
            THEN DATEDIFF(CURRENT_DATE(), status_date)
            ELSE NULL 
        END
    ), 2) AS avg_days_in_current_status
FROM status_buckets
GROUP BY status_desc, status_category
ORDER BY status_category ASC, pct_of_total DESC;

#Testing Code

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_avg_gap_sub_n_issue_cd;

In [0]:
%sql
SELECT * 
FROM la_lakehouse.gold.vw_status_buckets;